# Vaani Track 1 — v2 (Kaggle, GPU T4 x2)

Span-regression pipeline: ATST-Frame + BEATs fusion, trident boundary head, count-head
selection, per-district calibration.

**Run order:** setup -> encoders -> data -> VAD + synthetic -> train -> diagnose -> submit.

Set the accelerator to **GPU T4 x2** and add your `HF_TOKEN` under *Add-ons -> Secrets*
(the dataset is gated). Kaggle has no Drive, so the corpus downloads fresh each session --
save `/kaggle/working/data` as a Kaggle Dataset once if you want it to persist.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/raut7218/vaani-sed-v2.git /kaggle/working/v2
%cd /kaggle/working/v2
!pip -q install -r requirements.txt


## 0. Verify the wiring before spending GPU hours

`test_overfit.py` is the one that matters: it proves the head is time-aligned to the audio. A silent offset between waveform and targets does not show up in the loss and costs the entire event-F1 term.

In [ ]:
!python tests/test_components.py
!python tests/test_overfit.py


## 1. Encoders

ATST-Frame (40 ms) is the primary encoder; BEATs (160 ms) rides along as a semantic channel. If the ATST weights fail to download, training still runs BEATs-only -- just less accurately.

In [ ]:
!python scripts/fetch_encoders.py --all
!ls -la checkpoints/


## 2. Data

Full corpus: 182 shards, ~16.5 GB parquet, 90,637 clips (~154.6 h). Use `--max-shards` for a quick smoke run first.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# quick check first; drop --max-shards for the full corpus
!python scripts/download_data.py --out /kaggle/working/data --max-shards 4


In [ ]:
!python scripts/download_data.py --out /kaggle/working/data
!python -c "import json;r=[json.loads(l) for l in open('/kaggle/working/data/manifest.jsonl',encoding='utf-8')];from collections import Counter;print(len(r),'clips');print(Counter(x['tier'] for x in r))"


## 3. Free supervision

**VAD** gives the speech head its pseudo-labels -- Vaani is *speech* recordings, and factoring speech out makes noise boundaries much easier to place.

**Synthetic** turns the bronze tier's 32 tag-only hours into strong labels: cut a segment from a clip tagged `animal_sound`, paste it at a known position, and the label is correct *by construction* -- no confidence threshold, so no error to accumulate.

In [ ]:
!python scripts/make_vad.py --data /kaggle/working/data
!python scripts/make_synthetic.py --data /kaggle/working/data --out /kaggle/working/synth -n 20000


## 4. Train

`--batch-size` is **per GPU** (standard DDP convention), so 16 x 2 = 32 global.

Train several folds if you have the budget -- selecting on one narrow state slice is what cost v1 0.16 between validation and the leaderboard.

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data']['vad_dir'] = '/kaggle/working/data/vad'
cfg['model']['beats_dir'] = '/kaggle/working/v2/checkpoints'
pathlib.Path('configs/kaggle.yaml').write_text(yaml.safe_dump(cfg))
print(open('configs/kaggle.yaml').read()[:600])


In [ ]:
!torchrun --standalone --nproc_per_node=2 -m src.train.train     --config configs/kaggle.yaml     --data /kaggle/working/data --extra-data /kaggle/working/synth     --out /kaggle/working/runs/f0 --fold 0 --batch-size 16


## 5. Diagnose

The headline score says almost nothing about *which* failure mode you are in. This prints the constant-baseline comparison, the strict-vs-loose recall split (detection vs localisation), the boundary error distribution, the operating point against the corpus prior, and the selection oracle.

In [ ]:
!python scripts/diagnose.py --ckpt /kaggle/working/runs/f0/best.pt     --data /kaggle/working/data --fold 0


## 6. More folds, then ensemble

Pass every checkpoint to `predict.py`: candidates are fused with **1D weighted box fusion**, not by averaging posteriors. Averaging two models that localise an onset 80 ms apart widens the ramp by 80 ms; fusing spans keeps the edges sharp.

In [ ]:
for fold in [1, 2]:
    !torchrun --standalone --nproc_per_node=2 -m src.train.train         --config configs/kaggle.yaml --data /kaggle/working/data         --extra-data /kaggle/working/synth         --out /kaggle/working/runs/f{fold} --fold {fold} --batch-size 16


## 7. Submit

Per-district calibration is transductive: it groups the test clips by the `State_District` encoded in their filenames and matches each group's predicted event count and coverage to the corpus prior. It uses only unlabelled test audio.

Watch the final line -- if `events/clip` and `coverage` are far from the priors, the operating point is wrong and that is worth more than any model change.

In [ ]:
!python -m src.infer.predict     --ckpt /kaggle/working/runs/f0/best.pt /kaggle/working/runs/f1/best.pt /kaggle/working/runs/f2/best.pt     --audio-dir /kaggle/input/<your-test-audio-dir>     --out /kaggle/working/submission.zip


In [ ]:
import zipfile, json
with zipfile.ZipFile('/kaggle/working/submission.zip') as z:
    assert z.namelist() == ['predictions.jsonl'], z.namelist()
    rows = [json.loads(l) for l in z.read('predictions.jsonl').decode().splitlines() if l.strip()]
print(len(rows), 'clips |', sum(len(r['events']) for r in rows), 'events')
print(rows[0])
